# APDIS — Pipeline Exploration
Run each cell to see every stage of the RAG pipeline on one document. Select the project's `.venv` as the kernel.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.utils import config
pdfs = sorted(config.RAW_DOCS_DIR.glob('*.pdf'))
pdfs

## 1. Extract text page by page

In [ ]:
from src.document_processing.pdf_loader import load_pdf
doc = load_pdf(pdfs[0])
print(doc.source, doc.num_pages, 'pages, empty:', doc.empty_pages)
print(doc.pages[0].text[:600])

## 2. Clean

In [ ]:
from src.document_processing.text_cleaner import clean_document, find_repeated_lines
print('Headers/footers removed:', find_repeated_lines([p.text for p in doc.pages]))
cleaned = clean_document(doc)
print(cleaned.pages[0].text[:600])

## 3. Chunk

In [ ]:
from src.document_processing.text_chunker import chunk_document
chunks = chunk_document(cleaned, config.CHUNK_SIZE, config.CHUNK_OVERLAP)
print(len(chunks), 'chunks')
for c in chunks[:3]:
    print(f'--- {c.id} | page {c.page} | {len(c.text)} chars\n{c.text[:300]}')

## 4. Embed (first run downloads the model, ~90 MB)

In [ ]:
from src.embeddings.embedding_model import get_embedding_model, cosine_similarity
emb = get_embedding_model()
vecs = emb.embed_documents([c.text for c in chunks])
print(vecs.shape, 'max tokens read:', emb.max_seq_length)
print(vecs[0][:8])

In [ ]:
q = emb.embed_query('What are the responsibilities of an internal auditor?')
scores = sorted(((cosine_similarity(q, v), c.page, c.text[:80]) for v, c in zip(vecs, chunks)), reverse=True)
for s in scores[:3]: print(round(s[0], 3), 'p', s[1], s[2])

## 5. Ask through the full assistant (retrieval mode, then Gemini)

In [ ]:
from src.chatbot.assistant import APDISAssistant
a = APDISAssistant(embedder=emb)
print(a.documents())
r = a.ask('What are the responsibilities of an internal auditor?', mode='retrieval', top_k=3)
print(r.answer[:800])

In [ ]:
r = a.ask('What are the responsibilities of an internal auditor?', mode='rag')
print(r.answer)
print('Sources:', [str(x) for x in r.references], '| model:', r.model)